In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset("Abdelrahman-Rezk/Arabic_Dialect_Identification")

Repo card metadata block was not found. Setting CardData to empty.


In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'text'],
        num_rows: 440052
    })
    validation: Dataset({
        features: ['id', 'label', 'text'],
        num_rows: 9164
    })
    test: Dataset({
        features: ['id', 'label', 'text'],
        num_rows: 8981
    })
})


In [6]:
print(dataset['train'].column_names)
print(dataset['train'].features)

['id', 'label', 'text']
{'id': Value('int64'), 'label': Value('int64'), 'text': Value('string')}


In [8]:
import pandas as pd
df = dataset['train'].to_pandas()
print(df['label'].value_counts())

label
10    55353
11    42010
3     40442
17    35053
4     29839
6     26816
5     26524
2     25769
15    25254
13    25251
0     18359
7     15599
14    15542
8     14883
1     13862
9     11082
12     9534
16     8880
Name: count, dtype: int64


In [10]:
df['label'].unique()

array([ 9, 17,  0, 11,  2,  4, 10, 16,  6, 15,  1,  3,  5, 12,  7, 13, 14,
        8])

In [11]:
label_names = ['OM', 'SD', 'SA', 'KW', 'QA', 'LB', 'JO', 'SY', 'IQ', 'MA', 'EG', 'PL', 'YE', 'BH', 'DZ', 'AE', 'TN', 'LY']

df['dialect'] = df['label'].apply(lambda x: label_names[x])
df.tail()

,id,label,text,dialect
440047,1024415002810306560,6,انت مش مجبور تجامل حدا وتعطيه اكبر من حجمه ؛ ع...,JO
440048,825422832138067968,3,رتم الفريق بدون جيرو سريييييييييع ، مع جيرو يص...,KW
440049,1137082760726810624,2,🇸🇦💚🇸🇦💚🇸🇦\nksa\n👆🏻\nوغير هالبلد قلبي ما عشق ولا...,SA
440050,1075003837436575616,11,@a7lawa7di @ranim2298 خلص اذا هيييك بطلت ادعي ...,PL
440051,1072373084320120832,4,@aalsubaie___ @Ahmedhalyami90 خيولي هالدلوع مر...,QA


In [14]:
df.isnull().sum()

id         0
label      0
text       0
dialect    0
dtype: int64

In [15]:
(df['text'].str.strip() == '').sum()

np.int64(0)

In [16]:
df['text_length'] = df['text'].str.len()
print(df['text_length'].describe())

count    440052.000000
mean         88.992771
std          56.356761
min           7.000000
25%          49.000000
50%          72.000000
75%         114.000000
max         823.000000
Name: text_length, dtype: float64


In [18]:
import re

def clean_tweet(text):
    text = re.sub(r'@\w+', '', text)        # remove @mentions
    text = re.sub(r'http\S+', '', text)      # remove URLs
    text = re.sub(r'#', '', text)            # remove # symbol, keep the word after it
    text = re.sub(r'\s+', ' ', text).strip() # collapse extra spaces left behind
    return text

df['text_clean'] = df['text'].apply(clean_tweet)

In [21]:
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

val_df['text_clean'] = val_df['text'].apply(clean_tweet)
test_df['text_clean'] = test_df['text'].apply(clean_tweet)

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=50000)
X_train = vectorizer.fit_transform(df['text_clean'])
X_val = vectorizer.transform(val_df['text_clean'])

clf = LinearSVC()
clf.fit(X_train, df['label'])

preds = clf.predict(X_val)
print(classification_report(val_df['label'], preds, target_names=label_names))
print("Macro F1:", f1_score(val_df['label'], preds, average='macro'))

              precision    recall  f1-score   support

          OM       0.45      0.36      0.40       382
          SD       0.71      0.58      0.64       289
          SA       0.43      0.49      0.46       537
          KW       0.53      0.63      0.57       842
          QA       0.53      0.54      0.53       621
          LB       0.62      0.72      0.67       552
          JO       0.47      0.36      0.41       558
          SY       0.47      0.38      0.42       325
          IQ       0.60      0.55      0.57       310
          MA       0.78      0.68      0.72       231
          EG       0.73      0.87      0.79      1153
          PL       0.53      0.56      0.54       875
          YE       0.41      0.22      0.28       198
          BH       0.48      0.39      0.43       526
          DZ       0.63      0.53      0.58       324
          AE       0.47      0.47      0.47       526
          TN       0.73      0.53      0.61       185
          LY       0.67    

In [25]:
from sklearn.metrics import f1_score

X_test = vectorizer.transform(test_df['text_clean'])
test_preds = clf.predict(X_test)
print("Baseline test macro F1:", round(f1_score(test_df['label'], test_preds, average='macro'), 4))

Baseline test macro F1: 0.524


In [26]:
import pandas as pd
from sklearn.metrics import classification_report

ev = pd.read_csv('~/Downloads/test_predictions.csv')

region = {'OM':'Gulf','SA':'Gulf','KW':'Gulf','QA':'Gulf','BH':'Gulf','AE':'Gulf','YE':'Gulf',
          'LB':'Levant','JO':'Levant','SY':'Levant','PL':'Levant',
          'EG':'Nile','SD':'Nile','IQ':'Iraq',
          'MA':'Maghreb','DZ':'Maghreb','TN':'Maghreb','LY':'Maghreb'}

iq = ev[ev['true'] == 'IQ']
print(len(iq), (iq['pred'] == 'IQ').sum())            # expect 304 and 181
print(iq[iq['pred'] != 'IQ']['pred'].value_counts())              # all wrong destinations
print(iq[iq['pred'] != 'IQ']['pred'].map(region).value_counts())  # by region

wrongly_iq = ev[(ev['pred'] == 'IQ') & (ev['true'] != 'IQ')]
print(wrongly_iq['true'].map(region).value_counts())              # who gets called Iraqi

print(classification_report(ev['true'], ev['pred'], digits=3))    # all 18 countries

304 181
pred
KW    22
BH    19
OM    11
SY     9
JO     8
QA     8
LY     7
DZ     7
PL     6
AE     5
LB     4
YE     4
TN     4
SA     4
EG     4
MA     1
Name: count, dtype: int64
pred
Gulf       73
Levant     27
Maghreb    19
Nile        4
Name: count, dtype: int64
true
Gulf       50
Levant     12
Maghreb    11
Nile        5
Name: count, dtype: int64
              precision    recall  f1-score   support

          AE      0.524     0.574     0.548       516
          BH      0.502     0.493     0.498       515
          DZ      0.699     0.653     0.675       317
          EG      0.816     0.901     0.856      1130
          IQ      0.699     0.595     0.643       304
          JO      0.532     0.528     0.530       547
          KW      0.674     0.657     0.665       825
          LB      0.758     0.754     0.756       541
          LY      0.797     0.792     0.795       716
          MA      0.789     0.646     0.710       226
          OM      0.523     0.507     0.515     

In [27]:
bad = ev[(ev['true'] == 'IQ') & (ev['pred'].map(region) == 'Gulf')]
print(len(bad))
sample = bad.sample(20, random_state=1)
for n, (_, r) in enumerate(sample.iterrows(), 1):
    print(n, r['text_clean'])
    print()

73
1 [مستخدم] [مستخدم] كريم علينا وعليكم . . الله يبارك فيك يارب

2 [مستخدم] ليش يخجل ! ! هو اللي مقاومهم وطردهم من بلاده تريده يخجل الان بالعكس هذه الاشكال تفتخر بهم لانها ذليله وعبده للمنصب والدولار

3 يالذابح الماي بعطش كل شي يروح وما رحت من صابك مثلث سهم صاح السهم يالما صحت # محرم _ الحسين

4 [مستخدم] [مستخدم] لا مو صحيح اسفه بس ما أوافقكم الرأي

5 [مستخدم] عجل مالك غير الشنينه وخلك عن الأتكيت 😄

6 [مستخدم] ما همني منو اساسي همي الوحيد ننافس على اللقب

7 كيف تعرف ان الوهابي حمار يترحم للي خلاهم يلبسون اقنعه حتى في الصلاه 👌 🏿 👌 🏿 👌 🏿 [رابط]

8 [مستخدم] [مستخدم] شوفي انا مدير قاعة والله اتحدى اذا اكو غش

9 اللي ما نهض من أجل رفع الظلم وللدفاع عن المسكين ينهض من أجل الطالب اللي ضربة معلمة 😄 [رابط]

10 [مستخدم] [مستخدم] نايمين ولا مره نمر بذكركم يالشعوب العربيه

11 [مستخدم] هذا عبقري زمانه المفروض نحنطه ونخليه بالمتحف الطبيعي 😂

12 [مستخدم] معليه انت قلتي بس اكل 😂 😂

13 [مستخدم] مشكور اخوي الله يبارك بعمرك

14 أتابع حسابات طبخ ولا مره طبقت طبخة من اللي ينزلوها وهم بعدني أتابعها بشغف

1

In [29]:
import torch, pandas as pd
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from arabert.preprocess import ArabertPreprocessor

label_names = ['OM','SD','SA','KW','QA','LB','JO','SY','IQ','MA','EG','PL','YE','BH','DZ','AE','TN','LY']
cfg = AutoConfig.from_pretrained("final_model")
cfg.id2label = {i: n for i, n in enumerate(label_names)}
cfg.label2id = {n: i for i, n in enumerate(label_names)}
cfg.save_pretrained("final_model")

tok = AutoTokenizer.from_pretrained("final_model")
model = AutoModelForSequenceClassification.from_pretrained("final_model").eval()
prep = ArabertPreprocessor(model_name="aubmindlab/bert-base-arabertv02-twitter")

ev = pd.read_csv('~/Downloads/test_predictions.csv').sample(200, random_state=0)

def predict_top1(text):
    enc = tok(prep.preprocess(text), return_tensors="pt", truncation=True, max_length=64)
    with torch.no_grad():
        return model.config.id2label[model(**enc).logits.argmax(-1).item()]

mine = pd.Series([predict_top1(t) for t in ev['text']], index=ev.index)
print("same as Kaggle:", (mine == ev['pred']).mean())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

same as Kaggle: 1.0
